# Silver Layer Notebook

In [0]:
#import
from pyspark.sql import functions as F

## Read Bronze tables

In [0]:
transactions_df = spark.table("pyspark_module.bronze.transactions_data")
cards_df = spark.table("pyspark_module.bronze.cards_data_batch")
users_df = spark.table("pyspark_module.bronze.users_data")
mcc_df = spark.table("pyspark_module.bronze.mcc_codes")
fraud_labels_df = spark.table("pyspark_module.bronze.train_fraud_labels")

## Clean fraud labels

In [0]:
fraud_clean_df = (
    fraud_labels_df
    .withColumn("transaction_id", F.col("transaction_id").cast("int"))
    .withColumn(
        "is_fraud",
        F.when(F.lower(F.trim(F.col("is_fraud_raw"))).isin("yes", "true", "1"), F.lit(True))
         .otherwise(F.lit(False))
    )
    .select("transaction_id", "is_fraud")
)

## Clean MCC codes

In [0]:
mcc_clean_df = (
    mcc_df
    .withColumn("mcc", F.col("mcc_code").cast("int"))
    .withColumnRenamed("description", "mcc_description")
    .select("mcc", "mcc_description")
)

## Create Silver transactions

In [0]:
silver_transactions_df = (
    transactions_df
    .withColumnRenamed("id", "transaction_id")
    .withColumn("transaction_id", F.col("transaction_id").cast("int"))
    .withColumn("client_id", F.col("client_id").cast("int"))
    .withColumn("card_id", F.col("card_id").cast("int"))
    .withColumn("merchant_id", F.col("merchant_id").cast("int"))
    .withColumn("mcc", F.col("mcc").cast("int"))
    .withColumn("amount", F.regexp_replace(F.col("amount"), "[$,]", "").cast("double"))
    .withColumn("transaction_timestamp", F.to_timestamp("date"))
    .withColumn("transaction_date", F.to_date("transaction_timestamp"))
    .withColumn("transaction_week", F.weekofyear("transaction_timestamp"))
    .withColumn("transaction_month", F.date_format("transaction_timestamp", "yyyy-MM"))
    .withColumn("day_of_week", F.date_format("transaction_timestamp", "EEEE"))
    .withColumn("hour_of_day", F.hour("transaction_timestamp"))
    .withColumn(
        "time_of_day",
        F.when(F.col("hour_of_day").between(5, 11), "Morning")
         .when(F.col("hour_of_day").between(12, 16), "Afternoon")
         .when(F.col("hour_of_day").between(17, 21), "Evening")
         .otherwise("Night")
    )
    .join(fraud_clean_df, on="transaction_id", how="left")
    .join(mcc_clean_df, on="mcc", how="left")
    .fillna({"is_fraud": False})
)

In [0]:
# validation
display(silver_transactions_df.limit(20))
silver_transactions_df.printSchema()
print(silver_transactions_df.count())

mcc,transaction_id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,transaction_timestamp,transaction_date,transaction_week,transaction_month,day_of_week,hour_of_day,time_of_day,is_fraud,mcc_description
5812,7475565,2010-01-01 05:56:00,1840,3322,10.79,Swipe Transaction,27156,Beaverton,OR,97005.0,,2010-01-01T05:56:00.000Z,2010-01-01,53,2010-01,Friday,5,Morning,false,Eating Places and Restaurants
5499,7476526,2010-01-01 09:59:00,1771,5937,-77.0,Swipe Transaction,43293,Shingle Springs,CA,95682.0,,2010-01-01T09:59:00.000Z,2010-01-01,53,2010-01,Friday,9,Morning,false,Miscellaneous Food Stores
5814,7476271,2010-01-01 09:01:00,1525,4257,2.36,Swipe Transaction,44919,Arcadia,LA,71001.0,,2010-01-01T09:01:00.000Z,2010-01-01,53,2010-01,Friday,9,Morning,false,Fast Food Restaurants
5411,7476782,2010-01-01 10:52:00,112,1134,17.15,Swipe Transaction,50783,Atlanta,GA,30327.0,,2010-01-01T10:52:00.000Z,2010-01-01,53,2010-01,Friday,10,Morning,false,"Grocery Stores, Supermarkets"
5499,7475540,2010-01-01 05:38:00,1769,43,0.18,Swipe Transaction,86438,Manahawkin,NJ,8050.0,,2010-01-01T05:38:00.000Z,2010-01-01,53,2010-01,Friday,5,Morning,false,Miscellaneous Food Stores
5411,7475476,2010-01-01 03:32:00,1538,3206,14.36,Swipe Transaction,98374,Nazareth,PA,18064.0,,2010-01-01T03:32:00.000Z,2010-01-01,53,2010-01,Friday,3,Night,false,"Grocery Stores, Supermarkets"
5499,7475992,2010-01-01 07:48:00,1772,5918,12.5,Swipe Transaction,43293,El Monte,CA,91731.0,,2010-01-01T07:48:00.000Z,2010-01-01,53,2010-01,Friday,7,Morning,false,Miscellaneous Food Stores
4121,7476891,2010-01-01 11:16:00,1936,5551,25.92,Online Transaction,18563,ONLINE,,null,,2010-01-01T11:16:00.000Z,2010-01-01,53,2010-01,Friday,11,Morning,false,Taxicabs and Limousines
5499,7476196,2010-01-01 08:42:00,914,2859,57.0,Swipe Transaction,59935,Wichita,KS,67212.0,,2010-01-01T08:42:00.000Z,2010-01-01,53,2010-01,Friday,8,Morning,false,Miscellaneous Food Stores
5541,7476239,2010-01-01 08:55:00,277,4264,2.48,Swipe Transaction,61195,Grenada,MS,38901.0,,2010-01-01T08:55:00.000Z,2010-01-01,53,2010-01,Friday,8,Morning,false,Service Stations


root
 |-- mcc: integer (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: float (nullable = true)
 |-- errors: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_week: integer (nullable = true)
 |-- transaction_month: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- time_of_day: string (nullable = false)
 |-- is_fraud: boolean (nullable = false)
 |-- mcc_description: string (nullable = true)

30046


In [0]:
silver_transactions_df.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.silver.transactions"
)

## Create Silver cards

In [0]:
# removes sensitive fields like card number and CVV.
silver_cards_df = (
    cards_df
    .withColumn("card_id", F.col("id").cast("int"))
    .withColumn("client_id", F.col("client_id").cast("int"))
    .withColumn("credit_limit", F.regexp_replace(F.col("credit_limit"), "[$,]", "").cast("double"))
    .withColumn("acct_open_date", F.to_date("acct_open_date", "MM/yyyy"))
    .withColumn("expires", F.to_date("expires", "MM/yyyy"))
    .drop("id", "card_number", "cvv")
)

In [0]:
# validate
display(silver_cards_df.limit(20))
silver_cards_df.printSchema()
print(silver_cards_df.count())

client_id,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,card_id
825,Visa,Debit,2022-12-01,YES,2,24295.0,2002-09-01,2008,No,4524
825,Visa,Debit,2020-12-01,YES,2,21968.0,2014-04-01,2014,No,2731
825,Visa,Debit,2024-02-01,YES,2,46414.0,2003-07-01,2004,No,3701
825,Visa,Credit,2024-08-01,NO,1,12400.0,2003-01-01,2012,No,42
825,Mastercard,Debit (Prepaid),2009-03-01,YES,1,28.0,2008-09-01,2009,No,4659
1746,Visa,Credit,2003-09-01,YES,1,27500.0,2003-09-01,2012,No,4537
1746,Visa,Debit,2022-07-01,YES,2,28508.0,2011-02-01,2011,No,1278
1746,Mastercard,Debit,2022-06-01,YES,2,9022.0,2003-07-01,2015,No,3687
1746,Mastercard,Debit (Prepaid),2020-11-01,YES,2,54.0,2010-06-01,2015,No,3465
1746,Mastercard,Debit (Prepaid),2023-02-01,YES,1,99.0,2006-07-01,2012,No,3754


root
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- expires: date (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: double (nullable = true)
 |-- acct_open_date: date (nullable = true)
 |-- year_pin_last_changed: integer (nullable = true)
 |-- card_on_dark_web: string (nullable = true)
 |-- card_id: integer (nullable = true)

6146


In [0]:
# write
silver_cards_df.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.silver.cards"
)

## Create Silver users

In [0]:
silver_users_df = (
    users_df
    .withColumn("client_id", F.col("id").cast("int"))
    .withColumn("current_age", F.col("current_age").cast("int"))
    .withColumn("retirement_age", F.col("retirement_age").cast("int"))
    .withColumn("birth_year", F.col("birth_year").cast("int"))
    .withColumn("birth_month", F.col("birth_month").cast("int"))
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("per_capita_income", F.regexp_replace(F.col("per_capita_income"), "[$,]", "").cast("double"))
    .withColumn("yearly_income", F.regexp_replace(F.col("yearly_income"), "[$,]", "").cast("double"))
    .withColumn("total_debt", F.regexp_replace(F.col("total_debt"), "[$,]", "").cast("double"))
    .withColumn("credit_score", F.col("credit_score").cast("int"))
    .withColumn("num_credit_cards", F.col("num_credit_cards").cast("int"))
    .drop("id")
)

In [0]:
# validate
display(silver_users_df.limit(20))
silver_users_df.printSchema()
print(silver_users_df.count())

current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,client_id
53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5,825
53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5,1746
81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5,1718
63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4,708
43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1,1164
42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,20599.0,41997.0,0.0,704,3,68
36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,25258.0,51500.0,102286.0,672,3,1075
26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,26790.0,54623.0,114711.0,728,1,1711
81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,26273.0,42509.0,2895.0,755,5,1116
34,60,1986,1,Female,887 Grant Street,29.97,-92.12,18730.0,38190.0,81262.0,810,1,1752


root
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: double (nullable = true)
 |-- yearly_income: double (nullable = true)
 |-- total_debt: double (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)
 |-- client_id: integer (nullable = true)

2000


In [0]:
# write the silver
silver_users_df.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.silver.users"
)